In [1]:
import pandas as pd
from ftfy import fix_text
import re
import json
import numpy as np
pd.set_option('display.max_colwidth', None)

In [5]:
df1 = pd.read_csv('gem25pro-cpt_1.csv')
df2 = pd.read_csv('gem25pro-cpt_2.csv')
df3 = pd.read_csv('gem25pro-cpt_3.csv')
df4 = pd.read_csv('gem25pro-cpt_4.csv')
df = pd.concat([df1, df2, df3, df4], ignore_index=True)
len(df)

1000

In [10]:
df.loc[df['ID'] == 'i_1981', 'predict'] = 'Opposite meaning'

In [11]:
labels = ['Opposite meaning', 'Misrepresentation', 'Related but unverifiable',
          'Entailment', 'Entity error', 'Unrelated and unverifiable',
          'Numeric error', 'Missing information']

def extract_label(text):
    if not isinstance(text, str):
        return 'MULTI_OR_NONE'

    # Thử parse JSON
    try:
        data = json.loads(text)
        if isinstance(data, dict) and 'answer' in data and data['answer'] in labels:
            return data['answer']
    except json.JSONDecodeError:
        pass  # Nếu không phải JSON, tiếp tục xử lý như thường

    # Nếu không phải JSON, tìm nhãn trong chuỗi
    found = [label for label in labels if label in text]
    if len(found) == 1:
        return found[0]
    else:
        return 'MULTI_OR_NONE'

# Áp dụng hàm
df['predict'] = df['predict'].apply(extract_label)
df_ambiguous = df[df['predict'] == 'MULTI_OR_NONE']
df = df[df['predict'] != 'MULTI_OR_NONE']
print(len(df_ambiguous) == 0)
print(len(df))
df_ambiguous.head(1)

True
1000


,ID,claim_clean,reference_clean,predict


In [4]:
df_ambiguous.to_csv('fix.csv', index=False)

In [12]:
df = df.rename(columns={'predict': 'Label'})

# Thay thế các giá trị trong cột 'Label' bằng dạng viết tắt
df['Label'] = df['Label'].replace({
    'Misrepresentation': 'misinter', 
    'Opposite meaning': 'negat',
    'Related but unverifiable': 'relunvef', 
    'Entailment': 'entail', 
    'Entity error': 'entierr',
    'Unrelated and unverifiable': 'unrelunvef', 
    'Numeric error': 'numerr',
    'Missing information': 'missinfo'
})

In [13]:
df = df.drop(columns=['claim_clean', 'reference_clean'], axis=1)
df['Label'].unique()

array(['misinter', 'negat', 'relunvef', 'entail', 'entierr', 'missinfo',
       'unrelunvef', 'numerr'], dtype=object)

In [14]:
df.to_csv('./gem25pro-cpt.csv', index=False)

In [15]:
df['Label'] = df['Label'].replace({
    'misinter': 'contra', 
    'negat': 'contra',
    'relunvef': 'unver', 
    'entail': 'entail', 
    'entierr': 'contra',
    'unrelunvef': 'unver', 
    'numerr': 'contra',
    'missinfo': 'contra'
})

In [16]:
df.to_csv('./gem25pro-cpt2.csv', index=False)